In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

class ReadinessAssessor:
    """
    Module 2: Đánh giá mức độ sẵn sàng Số và AI cho Vùng & Ngành (AIDEOM-VN).
    Kỹ thuật: TOPSIS kết hợp trọng số khách quan Entropy.
    """
    def __init__(self):
        # Xác định đường dẫn thư mục data từ thư mục notebooks (GIỮ NGUYÊN)
        self.data_dir = Path.cwd().parent / 'data'

    def load_csv(self, filename):
        """Nạp dữ liệu an toàn từ thư mục data."""
        file_path = self.data_dir / filename
        if not file_path.exists():
            raise FileNotFoundError(f"Không tìm thấy tệp: {file_path}")
        return pd.read_csv(file_path)

    def calculate_entropy_weights(self, matrix):
        """
        Tính trọng số Entropy khách quan cho ma trận quyết định.
        Mục đích: Tiêu chí nào có sự biến động dữ liệu càng lớn thì trọng số càng cao.
        """
        # Tránh chia cho 0 nếu có cột nào tổng bằng 0
        col_sums = matrix.sum(axis=0)
        col_sums[col_sums == 0] = 1e-12
        
        P = matrix / col_sums
        k = 1.0 / np.log(len(matrix))
        # Công thức Entropy: E = -k * sum(P * ln(P))
        entropy = -k * np.nansum(P * np.log(P + 1e-12), axis=0)
        dispersion = 1 - entropy
        return dispersion / dispersion.sum()

    def run_topsis(self, df, criteria, is_benefit):
        """
        Thuật toán TOPSIS để xếp hạng các phương án.
        """
        X = df[criteria].values.astype(float)
        
        # --- BƯỚC SỬA LỖI: CHUẨN HÓA MIN-MAX CHO ENTROPY ---
        # Đảm bảo dữ liệu không bị âm (như vụ Khai khoáng tăng trưởng -1.2%)
        X_entropy = np.zeros_like(X)
        for j in range(X.shape[1]):
            col_min = X[:, j].min()
            col_max = X[:, j].max()
            if col_max == col_min:
                X_entropy[:, j] = 1.0  # Xử lý trường hợp cả cột giống nhau hệt
            elif is_benefit[j]:
                X_entropy[:, j] = (X[:, j] - col_min) / (col_max - col_min)
            else:
                X_entropy[:, j] = (col_max - X[:, j]) / (col_max - col_min)
        
        # 1. Tính trọng số Entropy (Dùng ma trận Min-Max để không bị lỗi log)
        weights = self.calculate_entropy_weights(X_entropy)
        
        # 2. Chuẩn hóa ma trận cho TOPSIS (Vector Normalization - Giữ nguyên công thức cũ)
        norm_X = X / np.sqrt((X**2).sum(axis=0))
        
        # 3. Nhân trọng số
        weighted_X = norm_X * weights
        
        # 4. Xác định lời giải lý tưởng (A*) và lý tưởng âm (A-)
        ideal_star = np.where(is_benefit, weighted_X.max(axis=0), weighted_X.min(axis=0))
        ideal_neg = np.where(is_benefit, weighted_X.min(axis=0), weighted_X.max(axis=0))
        
        # 5. Tính khoảng cách Euclide
        dist_star = np.sqrt(((weighted_X - ideal_star)**2).sum(axis=1))
        dist_neg = np.sqrt(((weighted_X - ideal_neg)**2).sum(axis=1))
        
        # 6. Tính hệ số gần gũi tương đối C*
        closeness = dist_neg / (dist_star + dist_neg)
        return closeness, weights

# --- Thực thi Module 2 ---
if __name__ == "__main__":
    assessor = ReadinessAssessor()
    
    # --- PHẦN 1: ĐÁNH GIÁ 6 VÙNG KINH TẾ ---
    df_regions = assessor.load_csv('vietnam_regions_2024.csv')
    
    # Các tiêu chí đánh giá vùng
    reg_criteria = [
        'grdp_per_capita_million_VND', 'fdi_registered_billion_USD', 
        'digital_index_0_100', 'ai_readiness_0_100', 
        'trained_labor_pct', 'rd_intensity_pct', 'internet_penetration_pct', 'gini_coef'
    ]
    # Gini là chỉ số 'cost' (càng thấp càng tốt), các chỉ số khác là 'benefit'
    reg_benefit = [True, True, True, True, True, True, True, False]
    
    score_reg, w_reg = assessor.run_topsis(df_regions, reg_criteria, reg_benefit)
    df_regions['Readiness_Score'] = score_reg
    
    print("="*65)
    print(f"{'XẾP HẠNG SẴN SÀNG SỐ 6 VÙNG KINH TẾ (M2)':^65}")
    print("-"*65)
    print(df_regions[['region_name_en', 'Readiness_Score']].sort_values('Readiness_Score', ascending=False).to_string(index=False))

    # --- PHẦN 2: ĐÁNH GIÁ 10 NGÀNH KINH TẾ ---
    df_sectors = assessor.load_csv('vietnam_sectors_2024.csv')
    
    # Các tiêu chí đánh giá ngành
    sec_criteria = [
        'digital_index_0_100', 'ai_readiness_0_100', 'growth_rate_2024_pct',
        'export_billion_USD', 'spillover_coef_0_1', 'automation_risk_pct'
    ]
    # Rủi ro tự động hóa là 'cost'
    sec_benefit = [True, True, True, True, True, False]
    
    score_sec, w_sec = assessor.run_topsis(df_sectors, sec_criteria, sec_benefit)
    df_sectors['Priority_Score'] = score_sec
    
    print("\n" + "="*65)
    print(f"{'XẾP HẠNG ƯU TIÊN 10 NGÀNH KINH TẾ (M2)':^65}")
    print("-"*65)
    print(df_sectors[['sector_name_en', 'Priority_Score']].sort_values('Priority_Score', ascending=False).to_string(index=False))

            XẾP HẠNG SẴN SÀNG SỐ 6 VÙNG KINH TẾ (M2)             
-----------------------------------------------------------------
                       region_name_en  Readiness_Score
                            Southeast         0.927480
                      Red River Delta         0.920332
North Central and South Central Coast         0.343884
                         Mekong Delta         0.141072
      Northern Midlands and Mountains         0.103715
                    Central Highlands         0.035918

             XẾP HẠNG ƯU TIÊN 10 NGÀNH KINH TẾ (M2)              
-----------------------------------------------------------------
                 sector_name_en  Priority_Score
                  Manufacturing        0.926716
   Information-Communication-IT        0.620876
   Agriculture-Forestry-Fishery        0.154039
      Finance-Banking-Insurance        0.122288
                     Healthcare        0.099452
Logistics-Transport-Warehousing        0.098145
             E